# Step 3b — Dataset-independent static features


In [ ]:
# Colab has numpy/pandas/sklearn preinstalled — only install extras

In [ ]:
# Mount Drive + set PROJECT_ROOT
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/text-difficulty-classification'
except Exception:
    PROJECT_ROOT = os.path.abspath('.')
os.environ['PROJECT_ROOT'] = PROJECT_ROOT
print('PROJECT_ROOT =', PROJECT_ROOT)


Mounted at /content/drive
PROJECT_ROOT = /content/drive/MyDrive/text-difficulty-classification


In [ ]:
import argparse
import glob
import os
import re
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = os.environ['PROJECT_ROOT']
MULTI_DIR    = os.path.join(PROJECT_ROOT, 'outputs', 'multi_corpus')
OUT_DIR      = os.path.join(PROJECT_ROOT, 'outputs', 'static_metrics_generic')
os.makedirs(OUT_DIR, exist_ok=True)

LABEL_COL    = 'education_level'
META_COLS    = ['education_level', 'source_dataset', 'domain',
                'label_source', 'subject', 'split']


# ---------- text utilities ----------

_SENT_RE = re.compile(r'[.!?]+')
_WORD_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")
_SYLLABLE_RE = re.compile(r'[aeiouy]+', re.IGNORECASE)


In [ ]:
def _sentences(text):
    parts = _SENT_RE.split(text)
    return [p.strip() for p in parts if p.strip()]


In [ ]:
def _words(text):
    return _WORD_RE.findall(text or '')


In [ ]:
def _syllables(word):
    """Cheap heuristic syllable count. Good enough for batch features without
    CMU Pronouncing Dictionary."""
    w = word.lower().rstrip('e')
    n = len(_SYLLABLE_RE.findall(w))
    return max(n, 1)


In [ ]:
# ---------- features ----------


In [ ]:
def compute_generic_features(text):
    """Per-text feature dict. ~30 corpus-agnostic features."""
    text = str(text or '')
    sents = _sentences(text)
    words = _words(text)
    n_chars = len(text)
    n_words = len(words)
    n_sents = max(len(sents), 1)
    word_lens = [len(w) for w in words]
    syll_counts = [_syllables(w) for w in words] if words else [1]

    avg_word_len = float(np.mean(word_lens)) if word_lens else 0.0
    avg_sent_len = n_words / n_sents
    avg_syll = float(np.mean(syll_counts))

    # Type-token ratio (lexical diversity).
    types = len(set(w.lower() for w in words))
    ttr = types / n_words if n_words else 0.0

    # Frequency of long / complex words.
    n_long_words = sum(1 for w in word_lens if w >= 7)
    n_complex_words = sum(1 for s in syll_counts if s >= 3)
    pct_long    = n_long_words    / n_words if n_words else 0.0
    pct_complex = n_complex_words / n_words if n_words else 0.0

    # Punctuation features.
    n_q = text.count('?')
    n_excl = text.count('!')
    n_comma = text.count(',')
    n_semi = text.count(';')
    n_paren = text.count('(') + text.count(')')

    # Paragraph + line structure.
    n_paragraphs = max(1, len([p for p in text.split('\n\n') if p.strip()]))
    n_lines      = max(1, len([l for l in text.split('\n')   if l.strip()]))

    # Choice / answer-pattern proxies (works across QA datasets).
    n_choice_markers = len(re.findall(r'^\s*\(?[A-Da-d]\)\s', text, re.M))
    has_explanation  = int(bool(re.search(r'\bbecause\b|\bthus\b|\btherefore\b', text, re.I)))
    has_question     = int('?' in text)

    # Classical readability formulas (closed-form, no external deps).
    fk_grade = 0.39 * avg_sent_len + 11.8 * avg_syll - 15.59
    fk_ease  = 206.835 - 1.015 * avg_sent_len - 84.6 * avg_syll
    smog = (1.043 * np.sqrt(n_complex_words * (30 / n_sents)) + 3.1291) if n_sents else 0.0
    # Coleman-Liau: CLI = 5.879 * (chars/words) - 29.587 * (sents/words) - 15.8
    # (= 0.0588 * letters_per_100_words - 0.296 * sents_per_100_words - 15.8)
    chars_per_word = n_chars / n_words if n_words else 0.0
    sents_per_word = n_sents / n_words if n_words else 0.0
    coleman_liau = 5.879 * chars_per_word - 29.587 * sents_per_word - 15.8
    ari = (4.71 * (n_chars / n_words if n_words else 0)
           + 0.5 * avg_sent_len - 21.43)
    gunning_fog = 0.4 * (avg_sent_len + 100 * pct_complex)

    # Distributional shape — entropy of word-length histogram.
    if word_lens:
        hist = Counter(word_lens)
        probs = np.array(list(hist.values())) / len(word_lens)
        wlen_entropy = float(-(probs * np.log2(probs + 1e-12)).sum())
    else:
        wlen_entropy = 0.0

    return {
        'gen_n_chars':        float(n_chars),
        'gen_n_words':        float(n_words),
        'gen_n_sents':        float(n_sents),
        'gen_avg_word_len':   avg_word_len,
        'gen_avg_sent_len':   avg_sent_len,
        'gen_avg_syll':       avg_syll,
        'gen_ttr':            ttr,
        'gen_pct_long':       pct_long,
        'gen_pct_complex':    pct_complex,
        'gen_n_paragraphs':   float(n_paragraphs),
        'gen_n_lines':        float(n_lines),
        'gen_n_q':            float(n_q),
        'gen_n_excl':         float(n_excl),
        'gen_n_comma':        float(n_comma),
        'gen_n_semi':         float(n_semi),
        'gen_n_paren':        float(n_paren),
        'gen_n_choice_markers': float(n_choice_markers),
        'gen_has_explanation':  float(has_explanation),
        'gen_has_question':     float(has_question),
        'gen_fk_grade':       fk_grade,
        'gen_fk_ease':        fk_ease,
        'gen_smog':           smog,
        'gen_coleman_liau':   coleman_liau,
        'gen_ari':            ari,
        'gen_gunning_fog':    gunning_fog,
        'gen_wlen_entropy':   wlen_entropy,
    }


In [ ]:
def compute_for_df(df, desc='generic'):
    feats = []
    for i in tqdm(range(len(df)), desc=desc):
        feats.append(compute_generic_features(df.iloc[i]['full_text']))
    out = pd.DataFrame(feats)
    out = out.replace([np.inf, -np.inf], 0).fillna(0)
    # Carry meta cols through unchanged.
    for c in META_COLS:
        if c in df.columns:
            out[c] = df[c].values
    return out


In [ ]:
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--splits', nargs='+', default=None,
                    help='CSV basenames in outputs/multi_corpus/ to process. '
                         'Default = all train/val/test/ood_*.csv.')
    # Colab kernel-launcher fills sys.argv with `-f /tmp/k.json`. Detect and
    # fall back to defaults so users don't have to override sys.argv manually.
    import sys as _sys
    if any(a.endswith('.json') or a.startswith('-f') for a in _sys.argv[1:]):
        args = ap.parse_args([])
        print("[step3b] running under Jupyter — processing all splits.")
    else:
        args = ap.parse_args()

    if args.splits:
        paths = [os.path.join(MULTI_DIR, f'{s}.csv') for s in args.splits]
    else:
        paths = sorted(glob.glob(os.path.join(MULTI_DIR, '*.csv')))

    if not paths:
        raise SystemExit(f"[step3b] No CSVs in {MULTI_DIR}. Run Step 2b first.")

    for p in paths:
        df = pd.read_csv(p)
        if 'full_text' not in df.columns:
            print(f"[step3b] skip {p} — no full_text column.")
            continue
        out = compute_for_df(df, desc=os.path.basename(p))
        out_path = os.path.join(OUT_DIR, os.path.basename(p).replace('.csv', '_static_generic.csv'))
        out.to_csv(out_path, index=False)
        print(f"  wrote {out_path}  shape={out.shape}")


In [ ]:
main()


[step3b] running under Jupyter — processing all splits.


ood_onestop.csv:   0%|          | 0/567 [00:00<?, ?it/s]

  wrote /content/drive/MyDrive/text-difficulty-classification/outputs/static_metrics_generic/ood_onestop_static_generic.csv  shape=(567, 32)


ood_openbookqa.csv:   0%|          | 0/1500 [00:00<?, ?it/s]

  wrote /content/drive/MyDrive/text-difficulty-classification/outputs/static_metrics_generic/ood_openbookqa_static_generic.csv  shape=(1500, 32)


...

  wrote /content/drive/MyDrive/text-difficulty-classification/outputs/static_metrics_generic/test_static_generic.csv  shape=(795, 32)


train.csv:   0%|          | 0/6371 [00:00<?, ?it/s]

  wrote /content/drive/MyDrive/text-difficulty-classification/outputs/static_metrics_generic/train_static_generic.csv  shape=(6371, 32)


val.csv:   0%|          | 0/795 [00:00<?, ?it/s]

  wrote /content/drive/MyDrive/text-difficulty-classification/outputs/static_metrics_generic/val_static_generic.csv  shape=(795, 32)
